In [22]:
import requests # for making http (web) requests
import pandas as pd # for working with tabular (spreadsheet) data
import csv # also for working with tabular data, in csv format
from time import sleep
import random


In [23]:
# https://www.congress.gov/search?q=%7B%22source%22%3A%22congrecord%22%2C%22search%22%3A%22gender%22%2C%22congress%22%3A%22119%22%7D
bills = pd.read_csv('../bill_data/records-gender-119.csv')
df = pd.DataFrame(bills)
df = df.drop_duplicates(subset=['Title'], keep='first')

# separate date into three columns: year, month, day
# add zero in front of month and day if they are single digit
df[['Month', 'Day', 'Year']] = df['Issue Date'].str.split('/', expand=True)
df['Month'] = df['Month'].str.zfill(2)
df['Day'] = df['Day'].str.zfill(2)

In [24]:
df

,Title,URL,Issue Date,Section,Volume,Number,Page,Month,Day,Year
0,INTRODUCTION OF THE REAL ID GENDER REQUIREMENT...,https://www.congress.gov/congressional-record/...,6/2/25,Extensions of Remarks,171,93,E489,06,02,25
1,Text of Senate Amendment 4632; Congressional R...,https://www.congress.gov/congressional-record/...,3/18/26,Senate,172,49,S1279,03,18,26
2,SENATE RESOLUTION 604--RECOGNIZING THAT IT IS ...,https://www.congress.gov/congressional-record/...,2/11/26,Senate,172,29,S580,02,11,26
3,Text of Senate Amendment 4470; Congressional R...,https://www.congress.gov/congressional-record/...,3/18/26,Senate,172,49,S1214,03,18,26
4,"SENATE RESOLUTION 643--DESIGNATING MARCH 12, 2...",https://www.congress.gov/congressional-record/...,3/16/26,Senate,172,47,S1057,03,16,26
...,...,...,...,...,...,...,...,...,...,...
483,"CONTINUING APPROPRIATIONS AND EXTENSIONS ACT, ...",https://www.congress.gov/congressional-record/...,11/12/25,House,171,191,H4609,11,12,25
484,TEXT OF AMENDMENTS; Congressional Record Vol. ...,https://www.congress.gov/congressional-record/...,6/30/25,Senate,171,113,S4089,06,30,25
485,"CONSOLIDATED APPROPRIATIONS ACT, 2026; Congres...",https://www.congress.gov/congressional-record/...,1/22/26,House,172,15,H1185,01,22,26
486,TEXT OF AMENDMENTS; Congressional Record Vol. ...,https://www.congress.gov/congressional-record/...,11/20/25,Senate,171,196,S8283,11,20,25


In [50]:
def add_one(n):
    new = n + 1
    url = f'https://www.congress.gov/119/crec/20{year}/{month}/{day}/{volume}/{number}/modified/CREC-20{year}-{month}-{day}-pt1-Pg{page}-{new}.htm'
    print(url, new)
    return url, new

In [62]:
gender_119 = []

for i in range(len(df)):
    year = df.iloc[i]['Year']
    month = df.iloc[i]['Month']
    day = df.iloc[i]['Day']
    volume = df.iloc[i]['Volume']
    number = df.iloc[i]['Number'] 
    page = df.iloc[i]['Page']

    url = f'https://www.congress.gov/119/crec/20{year}/{month}/{day}/{volume}/{number}/modified/CREC-20{year}-{month}-{day}-pt1-Pg{page}.htm'
    
    r = requests.get(url)
    
    n = 1
    while r.status_code == 404 and n < 8:
        url, n = add_one(n)
        r = requests.get(url)
        sleep(random.uniform(.1, 2))
        if r.status_code == 200:
            break

    # if r.status_code == 404:
    #     url = f'https://www.congress.gov/119/crec/20{year}/{month}/{day}/modified/CREC-20{year}-{month}-{day}-pt1-Pg{page}.htm'
    #     r = requests.get(url)
    #     sleep(random.uniform(.1, 2))
    #     if r.status_code == 200:
    #         break
    #     url, n = add_one_no_vol(n)
            
    if r.status_code == 200:
        print(f"success at row {i} with url: {url}")
        text = r.text
        gender_119.append(text)
    
    # sleep for a random amount of time between .1 and 3 seconds
    sleep(random.uniform(.1, 2))

https://www.congress.gov/119/crec/2025/06/02/171/93/modified/CREC-2025-06-02-pt1-PgE489-2.htm 2
success at row 0 with url: https://www.congress.gov/119/crec/2025/06/02/171/93/modified/CREC-2025-06-02-pt1-PgE489-2.htm
success at row 1 with url: https://www.congress.gov/119/crec/2026/03/18/172/49/modified/CREC-2026-03-18-pt1-PgS1279.htm
success at row 2 with url: https://www.congress.gov/119/crec/2026/02/11/172/29/modified/CREC-2026-02-11-pt1-PgS580.htm
success at row 3 with url: https://www.congress.gov/119/crec/2026/03/18/172/49/modified/CREC-2026-03-18-pt1-PgS1214.htm
success at row 4 with url: https://www.congress.gov/119/crec/2026/03/16/172/47/modified/CREC-2026-03-16-pt1-PgS1057.htm
success at row 5 with url: https://www.congress.gov/119/crec/2026/02/05/172/26/modified/CREC-2026-02-05-pt1-PgS513.htm
success at row 6 with url: https://www.congress.gov/119/crec/2026/05/20/172/86/modified/CREC-2026-05-20-pt1-PgH3652.htm
success at row 7 with url: https://www.congress.gov/119/crec/2025

In [63]:
len(gender_119)

468

In [64]:
gender_119[-1]

"<pre>\n\n\n[Page S8270]\nFrom the Congressional Record Online through the Government Publishing Office [<a href='https://www.gpo.gov'>www.gpo.gov</a>]\n\n\n\n\n     RECOGNIZING THE 100TH ANNIVERSARY OF THE CHILDREN&#x27;S MUSEUM OF \n                              INDIANAPOLIS\n\n&lt;bullet&gt; Mr. YOUNG. Mr. President, I rise today to honor the 100th \nanniversary of the Children&#x27;s Museum of Indianapolis.\n  Towards the end of the 1920s, a Canadian newspaper took note of a \ncollection in America&#x27;s Midwest. One of its reporters marveled, ``In \nIndianapolis, Indiana, there is a unique institution called the \nChildren&#x27;s Museum. . . . &#x27;&#x27;\n  Today, a century later, that institution is no less unique. It is now \nthe largest museum for children in the entire world and the fourth \noldest. But to describe it as solely a museum for children is a \nmisnomer. In the era of the museum&#x27;s founding, opportunities for young \npeople to learn and grow beyond the class

In [75]:
# save list to txt file
with open('../records_texts/gender_records_119.txt', 'w') as f:
    for item in gender_119:
        f.write(item)

In [76]:
type(gender_119)

list

In [93]:
for i in gender_119:
    i.replace('\n', ' ')

In [95]:
gender_119[:100]

["<pre>\n\n\n[Extensions of Remarks]\n[Page E489]\nFrom the Congressional Record Online through the Government Publishing Office [<a href='https://www.gpo.gov'>www.gpo.gov</a>]\n\n\n\n\n\n       INTRODUCTION OF THE REAL ID GENDER REQUIREMENT REFORM ACT\n\n                                 ______\n                                 \n\n                       HON. ELEANOR HOLMES NORTON\n\n                      of the district of columbia\n\n                    in the house of representatives\n\n                          Monday, June 2, 2025\n\n  Ms. NORTON. Mr. Speaker, today, I introduce the REAL ID Gender \nRequirement Reform Act, which would repeal the requirement in the REAL \nID Act that REAL ID-compliant licenses include gender. Instead, states \nwould decide whether to include gender on their respective REAL ID-\ncompliant licenses. I am pleased Representative Maxwell Frost is co-\nleading this bill.\n  Under this bill, if a state includes gender on its REAL ID-compliant \nlicenses, 

In [68]:
# handling \n and white space

cleaned = cleaned.replace('\n', ' ')
cleaned = cleaned.replace('      ', ' ')
cleaned = cleaned.replace('     ', ' ')
cleaned = cleaned.replace('    ', ' ')
cleaned = cleaned.replace('   ', ' ')
cleaned = cleaned.replace('  ', ' ')

In [91]:
# loading up the texts that we just saved
load = open('../records_texts/gender_records_119.txt', 'r')
data = load.read()
load.close()

In [92]:
type(data)

str

In [83]:
# The next step will be to clean the text of the html characters and other unwanted characters 
# and whitespace. 

# remove all the characters in the "take out" list by writing a
# loop that replaces those characters with an empty character, ''
def clean_up(text):
    take_out = ['_', '[', ']', '<pre>', '</pre>', "<a href='https://www.gpo.gov'>www.gpo.gov</a>"]
    for item in take_out:
        if item in text:
            text = text.replace(item, '')
    return text


cleaned = clean_up(data)

AttributeError: 'list' object has no attribute 'replace'

In [84]:
cleaned_list = []
for i in data:
    cleaned = clean_up(i)
    cleaned_list.append(cleaned)

In [73]:
cleaned[:1000]

'\n\n\nExtensions of Remarks\nPage E489\nFrom the Congressional Record Online through the Government Publishing Office \n\n\n\n\n\n       INTRODUCTION OF THE REAL ID GENDER REQUIREMENT REFORM ACT\n\n                                 \n                                 \n\n                       HON. ELEANOR HOLMES NORTON\n\n                      of the district of columbia\n\n                    in the house of representatives\n\n                          Monday, June 2, 2025\n\n  Ms. NORTON. Mr. Speaker, today, I introduce the REAL ID Gender \nRequirement Reform Act, which would repeal the requirement in the REAL \nID Act that REAL ID-compliant licenses include gender. Instead, states \nwould decide whether to include gender on their respective REAL ID-\ncompliant licenses. I am pleased Representative Maxwell Frost is co-\nleading this bill.\n  Under this bill, if a state includes gender on its REAL ID-compliant \nlicenses, the state must allow individuals to change the gender \ndesigna

In [69]:
cleaned[:1000]

' Extensions of Remarks Page E489 From the Congressional Record Online through the Government Publishing Office INTRODUCTION OF THE REAL ID GENDER REQUIREMENT REFORM ACT HON. ELEANOR HOLMES NORTON of the district of columbia in the house of representatives Monday, June 2, 2025 Ms. NORTON. Mr. Speaker, today, I introduce the REAL ID Gender Requirement Reform Act, which would repeal the requirement in the REAL ID Act that REAL ID-compliant licenses include gender. Instead, states would decide whether to include gender on their respective REAL ID- compliant licenses. I am pleased Representative Maxwell Frost is co- leading this bill. Under this bill, if a state includes gender on its REAL ID-compliant licenses, the state must allow individuals to change the gender designation on their license through self-attestation. It would also require states that include gender on their REAL ID-compliant licenses to have a neutral or other designation gender field, in addition to male or female. Unde

In [70]:
# save plain to a separate text file

# MAKE SURE TO CHANGE THE NAME OF THE FILE!
with open('../records_texts/cleaned_119.txt', 'w') as f:
    f.write(cleaned)